In [ ]:
# Step 1: Dependencies
!pip install -q gdown opencv-python-headless
print('Done!')


In [ ]:
# Step 2: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive Mounted!')


In [ ]:
# Step 3: Extract Dataset
import os, zipfile, glob
EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)
zips = glob.glob('/content/drive/MyDrive/dataset*.zip')
if zips:
    print(f'Extracting {zips[0]}...')
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction Done!')
else:
    print('ERROR: No dataset zip found!')


In [ ]:
# Step 4: Download SIDD Weights
import os
SIDD = '/content/NAFNet-SIDD-width64.pth'
if not os.path.exists(SIDD):
    !gdown --id 14Fht1QQJ2gMlk4N1ERCRuElg8JfjrWWR -O /content/NAFNet-SIDD-width64.pth
print('SIDD Ready:', os.path.exists(SIDD))


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, os

class LN2d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(1,c,1,1))
        self.b = nn.Parameter(torch.zeros(1,c,1,1))
        self.eps = eps
    def forward(self, x):
        m = x.mean(1, keepdim=True)
        v = (x - m).pow(2).mean(1, keepdim=True)
        return (x-m)/torch.sqrt(v+self.eps)*self.w + self.b

class Gate(nn.Module):
    def forward(self, x):
        a, b = x.chunk(2, dim=1); return a * b

class SCA(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Conv2d(c, c, 1)
    def forward(self, x): return x * self.fc(self.pool(x))

class NAFBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        dw = c * 2
        self.c1 = nn.Conv2d(c, dw, 1)
        self.c2 = nn.Conv2d(dw, dw, 3, padding=1, groups=dw)
        self.sg1= Gate()
        self.sca= SCA(dw//2)
        self.c3 = nn.Conv2d(dw//2, c, 1)
        self.n1 = LN2d(c)
        fn = c * 2
        self.c4 = nn.Conv2d(c, fn, 1)
        self.sg2= Gate()
        self.c5 = nn.Conv2d(fn//2, c, 1)
        self.n2 = LN2d(c)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma= nn.Parameter(torch.zeros(1, c, 1, 1))
    def forward(self, x):
        y = self.c3(self.sca(self.sg1(self.c2(self.c1(self.n1(x))))))
        x = x + y * self.beta
        y = self.c5(self.sg2(self.c4(self.n2(x))))
        return x + y * self.gamma

class NAFNet(nn.Module):
    def __init__(self, w=128, scale=2):
        super().__init__()
        self.scale = scale
        self.intro = nn.Conv2d(1, w, 3, padding=1)
        self.enc   = nn.ModuleList()
        self.dec   = nn.ModuleList()
        self.up    = nn.ModuleList()
        self.down  = nn.ModuleList()
        c = w
        for _ in [2, 2]:
            self.enc.append(nn.Sequential(NAFBlock(c), NAFBlock(c)))
            self.down.append(nn.Conv2d(c, c*2, 2, stride=2))
            c *= 2
        self.mid = nn.Sequential(NAFBlock(c), NAFBlock(c))
        for _ in [2, 2]:
            self.up.append(nn.ConvTranspose2d(c, c//2, 2, stride=2))
            c //= 2
            self.dec.append(nn.Sequential(NAFBlock(c), NAFBlock(c)))
        self.head = nn.Sequential(
            nn.Conv2d(w, w*scale**2, 3, padding=1),
            nn.PixelShuffle(scale),
            NAFBlock(w),
            nn.Conv2d(w, 1, 3, padding=1))
    def forward(self, inp):
        H, W = inp.shape[2], inp.shape[3]
        ph = (4 - H % 4) % 4
        pw = (4 - W % 4) % 4
        x = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp
        base = F.interpolate(x, scale_factor=self.scale, mode='bicubic', align_corners=False)
        x = self.intro(x)
        sk = []
        for e, d in zip(self.enc, self.down):
            x = e(x); sk.append(x); x = d(x)
        x = self.mid(x)
        for u, dc in zip(self.up, self.dec):
            x = u(x); x = x + sk.pop(); x = dc(x)
        x = self.head(x)
        x = x + base
        return x[:, :, :H*self.scale, :W*self.scale]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
student = NAFNet(w=128, scale=2).to(device)

SIDD = '/content/NAFNet-SIDD-width64.pth'
if os.path.exists(SIDD):
    ckpt = torch.load(SIDD, map_location='cpu')
    sd = ckpt.get('state_dict', ckpt.get('params', ckpt))
    msd = student.state_dict()
    ok = 0
    for k, v in sd.items():
        k2 = k.replace('module.', '').replace('net.', '')
        if k2 not in msd: continue
        tgt = msd[k2]
        if k2 == 'intro.weight' and v.shape[1] == 3:
            v = v.mean(dim=1, keepdim=True)
        if v.shape != tgt.shape:
            pad = torch.zeros_like(tgt)
            sl = tuple(slice(0, min(a, b)) for a, b in zip(v.shape, tgt.shape))
            pad[sl] = v[sl]; v = pad
        msd[k2] = v; ok += 1
    student.load_state_dict(msd, strict=False)
    print(f'Initialized {ok} weights from SIDD-64')


In [ ]:
# Step 6: High-Resolution Dataset Loader (256x256 HR Patch)
from torch.utils.data import Dataset, DataLoader
import random, numpy as np, glob

def norm_img(x):
    x = x.astype(np.float32); m = x.max()
    if x.dtype == np.uint8 or m > 5: x = x / (255 if m <= 255 else m)
    elif m > 1: x = x / m
    return np.clip(x, 0, 1)

class SemiconDS(Dataset):
    def __init__(self, root, ps=256, sc=2):
        self.ps = ps; self.sc = sc
        npy = (glob.glob(os.path.join(root, '*.npy')) +
               glob.glob(os.path.join(root, '**', '*.npy'), recursive=True))
        dm, gm = {}, {}
        for fp in npy:
            pn = fp.replace('\\', '/').lower()
            if '__macosx' in pn or 'noisy-lr/noisylr' in pn: continue
            fn = pn.split('/')[-1]
            if '/train/train/gt/' in pn: gm[fn] = fp
            elif '/train/train/noisylr/' in pn: dm[fn] = fp
        common = sorted(set(dm) & set(gm))
        self.D = [dm[k] for k in common]
        self.G = [gm[k] for k in common]
        print(f'Paired: {len(self.D)} training pairs | Patch Size: {ps}x{ps} HR')
    def __len__(self): return len(self.D)
    def __getitem__(self, i):
        d = norm_img(np.load(self.D[i]))
        g = norm_img(np.load(self.G[i]))
        if d.ndim == 3: d = d.squeeze()
        if g.ndim == 3: g = g.squeeze()
        h, w = d.shape; p = self.ps // self.sc
        if h > p and w > p:
            t = random.randint(0, h-p); l = random.randint(0, w-p)
            d = d[t:t+p, l:l+p]
            g = g[t*self.sc:(t+p)*self.sc, l*self.sc:(l+p)*self.sc]
        if random.random() > 0.5: d, g = np.fliplr(d).copy(), np.fliplr(g).copy()
        if random.random() > 0.5: d, g = np.flipud(d).copy(), np.flipud(g).copy()
        k = random.randint(0, 3)
        if k: d, g = np.rot90(d, k).copy(), np.rot90(g, k).copy()
        return torch.from_numpy(d).unsqueeze(0).float(), torch.from_numpy(g).unsqueeze(0).float()

dataset = SemiconDS('/content/dataset', ps=256)
dl = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)


In [ ]:
# Step 7: Train Master Loss (Charbonnier + 0.10*SSIM + 0.15*Sobel)
import torch.nn as nn, torch.nn.functional as F
from tqdm import tqdm

def charbonnier_loss(p, t, eps=1e-6):
    return torch.mean(torch.sqrt((p - t)**2 + eps))

def edge_loss(p, t):
    dev = p.device
    sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=dev).view(1,1,3,3)
    sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],  dtype=torch.float32, device=dev).view(1,1,3,3)
    b, c, h, w = p.shape
    p2 = p.view(b*c, 1, h, w); t2 = t.view(b*c, 1, h, w)
    pg = torch.sqrt(F.conv2d(p2, sx, padding=1)**2 + F.conv2d(p2, sy, padding=1)**2 + 1e-6)
    tg = torch.sqrt(F.conv2d(t2, sx, padding=1)**2 + F.conv2d(t2, sy, padding=1)**2 + 1e-6)
    return F.l1_loss(pg, tg)

def ssim_loss(p, t):
    mu_p = F.avg_pool2d(p, 3, 1, 1)
    mu_t = F.avg_pool2d(t, 3, 1, 1)
    sig_p = F.avg_pool2d(p*p, 3, 1, 1) - mu_p**2
    sig_t = F.avg_pool2d(t*t, 3, 1, 1) - mu_t**2
    sig_pt = F.avg_pool2d(p*t, 3, 1, 1) - mu_p*mu_t
    C1, C2 = 0.01**2, 0.03**2
    ssim = ((2*mu_p*mu_t + C1)*(2*sig_pt + C2)) / ((mu_p**2 + mu_t**2 + C1)*(sig_p + sig_t + C2))
    return 1.0 - ssim.mean()

def master_loss(pred, gt):
    chb = charbonnier_loss(pred, gt)
    ssm = ssim_loss(pred, gt)
    edg = edge_loss(pred, gt)
    return chb + 0.10 * ssm + 0.15 * edg

EPOCHS = 50; LR = 2e-4
opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')
os.makedirs('/content/weights', exist_ok=True)

best = float('inf'); student.train()
print('NAFNet-128 Master Trainer | PSNR Maximization & Brick Mortar Detail Recovery')
print('-' * 70)

for ep in range(1, EPOCHS + 1):
    tot = 0
    for d, g in tqdm(dl, desc=f'Ep {ep}/{EPOCHS}', leave=False):
        d, g = d.to(device), g.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            out = student(d)
            if out.shape != g.shape:
                out = F.interpolate(out, size=g.shape[2:], mode='bilinear', align_corners=False)
            loss = master_loss(out, g)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item()
    avg = tot / len(dl); sched.step()
    lr_now = sched.get_last_lr()[0]
    print(f'Epoch {ep:>2}/{EPOCHS} | Master Loss: {avg:.5f} | LR: {lr_now:.2e}')
    if avg < best:
        best = avg
        torch.save(student.state_dict(), '/content/weights/best_model.pth')
        print(f'  --> Saved best! loss={best:.5f}')

print(f'Training Complete! Best Master Loss: {best:.5f}')


In [ ]:
# Step 8: Download Best Model Checkpoint
from google.colab import files
files.download('/content/weights/best_model.pth')
